# TED Talk Tag Analysis

The goal here is simple: each TED talk has a list of tags assigned by curators, and we want to use those tags to put each talk into a broad theme category.

We try two different approaches. The first builds a keyword lookup table and picks whichever theme gets the most tag matches. The second finds the rarest tag in each talk and uses that as the label instead. At the end we look at where the second method goes wrong and why the first one holds up better.

## Imports

Most of what we need is pandas and matplotlib. The NLP libraries (spacy, nrclex, vaderSentiment) are imported for potential sentiment analysis later but aren't used in the theme assignment itself.

In [1]:
import kagglehub
import pandas as pd
import spacy
import os
import ast
from collections import Counter
from nrclex import NRCLex
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import matplotlib
matplotlib.use('WebAgg')
import matplotlib.pyplot as plt
%matplotlib inline

C:\Users\elzes\anaconda3\envs\project_sentiment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading the data

The dataset comes from Kaggle and has two files: `ted_main.csv` with talk metadata (title, speaker, tags, views, etc.) and `transcripts.csv` with the full talk text. We join them side by side, drop the URL column since it's not useful, and remove any rows that don't have a transcript.

In [2]:
# Download the TED Talks dataset from Kaggle
path = kagglehub.dataset_download("rounakbanik/ted-talks")
print("Dataset path:", path)

# Load and merge the two CSV files
main        = pd.read_csv(os.path.join(path, "ted_main.csv"))
transcripts = pd.read_csv(os.path.join(path, "transcripts.csv"))

df = pd.concat([main, transcripts], axis=1)
df = df.drop(columns=["url"])                              # not needed for analysis
df = df.dropna(subset=["transcript"]).reset_index(drop=True)  # keep only talks with a transcript

print(f"DataFrame shape: {df.shape}")

Dataset path: C:\Users\elzes\.cache\kagglehub\datasets\rounakbanik\ted-talks\versions\3
DataFrame shape: (2467, 17)


## Theme map

This is just a dictionary, each key is a theme name and the value is a list of TED tags that belong to it. We came up with 11 themes that cover most of what TED talks about: science, environment, politics, business, health, arts, psychology, education, space, philosophy, and general lifestyle.

A few very generic tags like "culture" and "society" show up under multiple themes on purpose, so they don't randomly tip the scale toward one category when a talk is hard to classify. Two themes such as "future" and "prediction" were left out entirely.

In [3]:
THEME_MAP = {
    "Science & Technology": [
        "science", "technology", "computers", "software", "code", "programming",
        "ai", "robots", "virtual reality", "internet", "web", "microsoft", "google",
        "online video", "hack", "demo", "interface design", "open-source", "wikipedia",
        "innovation", "invention", "engineering", "materials", "nanoscale", "chemistry",
        "string theory", "complexity", "simplicity", "rocket science", "telescopes",
        "drones", "biomechanics", "biomimicry", "animation", "gaming", "physics", "data"
    ],
    "Environment & Nature": [
        "environment", "climate change", "sustainability", "green", "pollution",
        "alternative energy", "solar energy", "wind energy", "energy", "water",
        "nature", "ecology", "biodiversity", "oceans", "fish", "animals", "birds",
        "insects", "ants", "bees", "plants", "garden", "trees", "agriculture",
        "natural resources", "natural disaster", "geology", "anthropocene",
        "asteroid", "mining", "plastic", "solar system"
    ],
    "Society & Politics": [
        "politics", "policy", "democracy", "government", "law", "corruption",
        "military", "war", "terrorism", "violence", "crime", "prison", "evil",
        "inequality", "race", "gender", "women", "activism", "social change",
        "global issues", "global development", "state-building", "united states",
        "europe", "africa", "asia", "china", "india", "brazil", "south america",
        "new york", "cities", "urban planning", "infrastructure", "community",
        "world cultures", "ancient world", "history", "human origins", "humanity",
        "culture", "society"
    ],
    "Business & Economics": [
        "business", "economics", "entrepreneur", "investment", "microfinance",
        "money", "marketing", "consumerism", "shopping", "work", "leadership",
        "productivity", "work-life balance", "goal-setting", "success", "innovation",
        "women in business", "telecom", "transportation", "cars", "infrastructure"
    ],
    "Health & Medicine": [
        "health", "medicine", "disease", "cancer", "aids", "ebola", "virus",
        "bacteria", "vaccines", "health care", "public health", "heart health",
        "obesity", "aging", "genetics", "dna", "biotech", "biology", "microbiology",
        "neuroscience", "brain", "cognitive science", "prosthetics", "illness",
        "depression", "mental health", "suicide", "alzheimer's", "bioethics",
        "astrobiology", "death"
    ],
    "Arts & Culture": [
        "art", "design", "architecture", "film", "movies", "music", "performance",
        "dance", "theater", "photography", "literature", "books", "poetry",
        "writing", "storytelling", "spoken word", "comedy", "humor", "animation",
        "typography", "fashion", "beauty", "origami", "museums", "library",
        "industrial design", "product design", "science and art", "performance art",
        "live music", "singer", "violin", "piano", "guitar", "cello", "vocals",
        "composing", "conducting", "ted brain trust", "culture", "society"
    ],
    "Psychology & Mind": [
        "psychology", "happiness", "motivation", "potential", "personal growth",
        "decision-making", "choice", "mind", "memory", "consciousness", "identity",
        "self", "personality", "introvert", "body language", "fear", "empathy",
        "compassion", "love", "relationships", "evolutionary psychology",
        "behavior", "morality", "curiosity", "play", "creativity"
    ],
    "Education & Learning": [
        "education", "teaching", "children", "parenting", "youth", "learning",
        "wunderkind", "presentation", "communication", "language", "math",
        "statistics", "visualizations", "macarthur grant", "ted prize", "ted fellows",
        "tedx", "interview", "society"
    ],
    "Space & Exploration": [
        "space", "cosmos", "universe", "astronomy", "nasa", "moon", "mars",
        "planets", "flight", "aircraft", "exploration", "adventure", "travel",
        "dark matter", "big bang", "extraterrestrial life", "submarine", "physics"
    ],
    "Religion & Philosophy": [
        "religion", "christianity", "god", "atheism", "buddhism", "faith",
        "philosophy", "evolution", "life", "time", "charter for compassion",
        "ancient world"
    ],
    "Society & Lifestyle": [
        "food", "sex", "family", "sports", "extreme sports", "social media",
        "news", "media", "entertainment", "map", "smell", "senses", "illusion",
        "magic", "meme", "toy", "materials", "narcotics", "sociology",
        "anthropology", "archaeology", "paleontology", "dinosaurs", "primates",
        "apes", "poverty", "philanthropy", "disaster relief", "peace",
        "collaboration", "human origins", "intelligence", "culture", "society"
    ],
}


## Tag parsing and assignment logic

The tags column stores lists as plain text strings, so we first convert them back to real lists using ast.literal_eval. Then we flatten everything into one big list to count how often each tag appears across the whole dataset, we need that for the rarest-tag method.

assign_theme goes through each tag and checks if it appears in any theme's keyword list. It sums up the matches and returns whichever theme scored highest. If nothing matched, it returns "Other".

assign_rarest_theme skips the keyword map entirely. It just looks at a talk's own tags and returns whichever one appeared least often in the full dataset. The thinking is that rarer tags might be more specific and descriptive but as we'll see, that doesn't always play out well.

In [4]:
# Tags are stored as plain text strings, so we parse them back to lists
df["tags"] = df["tags"].apply(ast.literal_eval)

# Flatten all tags into one list and count frequencies
all_tags = [tag.lower() for tags in df["tags"] for tag in tags]
tag_counts = Counter(all_tags)

print(f"Unique tags: {len(tag_counts)}")


Unique tags: 415


In [5]:
def assign_theme(tags):
    scores = {theme: 0 for theme in THEME_MAP}
    for tag in tags:
        tag = tag.lower()
        for theme, keywords in THEME_MAP.items():
            if tag in keywords:
                scores[theme] += 1
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "Other"


def assign_rarest_theme(tags):
    if not tags:
        return "Other"
    return min(tags, key=lambda t: tag_counts.get(t.lower(), 0))

def assign_rarest_theme(tags):
    if not tags: return "Other"
    if len(tags) == 1: return tags[0]
    scores = {}
    for tag in map(str.lower, tags):
        scores[tag] = scores.get(tag, 0) + 1
    return min(scores, key=scores.get)


## Applying both methods

We run both functions on every row and save the results as new columns. The value counts on assigned_theme let us quickly check if anything is falling into "Other" a lot of those would mean the keyword map has gaps.

In [6]:
df["assigned_theme"]        = df["tags"].apply(assign_theme)
df["assigned_rarest_theme"] = df["tags"].apply(assign_rarest_theme)

# Theme distribution
print("Theme distribution (keyword voting):")
print(df["assigned_theme"].value_counts())

unmatched = df[df["assigned_theme"] == "Other"]
print(f"\nUnmatched talks: {len(unmatched)}")

Theme distribution (keyword voting):
assigned_theme
Society & Politics       614
Science & Technology     511
Arts & Culture           443
Health & Medicine        235
Environment & Nature     206
Business & Economics     133
Education & Learning      93
Psychology & Mind         88
Society & Lifestyle       74
Space & Exploration       47
Religion & Philosophy     23
Name: count, dtype: int64

Unmatched talks: 0


In [7]:
# Side-by-side comparison of both methods
df[["title", "name", "assigned_theme", "assigned_rarest_theme", "description"]]

,title,name,assigned_theme,assigned_rarest_theme,description
0,Do schools kill creativity?,Ken Robinson: Do schools kill creativity?,Education & Learning,children,Sir Ken Robinson makes an entertaining and pro...
1,Averting the climate crisis,Al Gore: Averting the climate crisis,Environment & Nature,alternative energy,With the same humor and humanity he exuded in ...
2,Simplicity sells,David Pogue: Simplicity sells,Science & Technology,computers,New York Times columnist David Pogue takes aim...
3,Greening the ghetto,Majora Carter: Greening the ghetto,Society & Politics,macarthur grant,"In an emotionally charged talk, MacArthur-winn..."
4,The best stats you've ever seen,Hans Rosling: The best stats you've ever seen,Society & Politics,africa,You've never seen data presented like this. Wi...
...,...,...,...,...,...
2462,Don't fear intelligent machines. Work with them,Garry Kasparov: Don't fear intelligent machine...,Science & Technology,ai,We must face our fears if we want to get the m...
2463,Am I not human? A call for criminal justice re...,Marlon Peterson: Am I not human? A call for cr...,Society & Politics,criminal justice,For a crime he committed in his early twenties...
2464,No one should die because they live too far fr...,Raj Panjabi: No one should die because they li...,Health & Medicine,africa,Illness is universal -- but access to care is ...
2465,Songs that bring history to life,Rhiannon Giddens: Songs that bring history to ...,Arts & Culture,history,Rhiannon Giddens pours the emotional weight of...


## Where the rarest-tag method breaks down

Looking at the results, the rarest-tag approach has a clear problem: it picks tags that are uncommon in the dataset, but uncommon doesn't mean meaningful.

Some examples:

| Talk | Keyword theme | Rarest tag |
|------|--------------|------------|
| *Do schools kill creativity?* | Education & Learning | `children` |
| *Averting the climate crisis* | Environment & Nature | `alternative energy` |
| *Greening the ghetto* | Environment & Nature | `macarthur grant` |
| *The best stats you've ever seen* | Science & Technology | `africa` |

In the first case,children is a fine tag but it's not really what the talk is about. In the third, macarthur grant is an award that got tagged onto the talk, it has nothing to do with the topic. And africa in the last one is just one data point Hans Rosling mentions, not the subject of the talk.

The keyword voting method avoids this because it looks at all the tags together rather than picking one outlier. The rarest-tag approach could still be interesting for finding niche subtopics, but it's not reliable enough to use as a main category label.